## imports

In [84]:
import numpy as np
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt 
import os


## Getting the World happiness Data

1. Downloaded the world hapiness data and convered it into one big padnas dictionary using concat(), can do this since all columns will be the same so it just stacks them 
2. removed individua indexes as we would have multiple rows labled as 0....
3. 

In [85]:
years = {
    2026: "Happiness_Data/WHR26_Data_Figure_2.1.xlsx",
    2025: "Happiness_Data/WHR25_Data_Figure_2.1v3.xlsx",
    2024: "Happiness_Data/WHR24_Data_Figure_2.1.xls",
    2023: "Happiness_Data/WHR23_Data_Figure_2.1.xls",
    2022: "Happiness_Data/WHR22_Data_Figure_2.1.xls",
}

happiness_frames = []
for year, path in years.items():
    df = pd.read_excel(path)
    df['report_year'] = year
    happiness_frames.append(df)

happiness = pd.concat(happiness_frames, ignore_index=True)
happiness.head()



,Year,Rank,Country name,Life evaluation (3-year average),Lower whisker,Upper whisker,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,...,Generosity,Perceptions of corruption,Ladder score in Dystopia,RANK,Country,Happiness score,Whisker-high,Whisker-low,Dystopia (1.83) + residual,Explained by: GDP per capita
0,2025.0,1.0,Finland,7.764,7.690,7.837,1.915,1.638,0.939,1.105,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025.0,2.0,Iceland,7.540,7.449,7.630,1.971,1.720,0.996,1.105,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025.0,3.0,Denmark,7.539,7.446,7.631,1.986,1.633,0.930,1.081,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025.0,4.0,Costa Rica,7.439,7.356,7.522,1.697,1.483,0.739,1.101,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025.0,5.0,Sweden,7.255,7.172,7.337,1.950,1.570,1.027,1.070,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data Cleaning

In [86]:
#happiness.info()

In [87]:
#happiness.describe()

In [88]:
if happiness.duplicated().sum() == 0:
    print("No Duplicates")
else:
    print(happiness.duplicated().sum())


No Duplicates


In [89]:
happiness.isnull().sum()

Year                                           427
Rank                                           427
Country name                                   147
Life evaluation (3-year average)               427
Lower whisker                                 2615
Upper whisker                                 2615
Explained by: Log GDP per capita              2344
Explained by: Social support                  2198
Explained by: Healthy life expectancy         2204
Explained by: Freedom to make life choices    2201
Explained by: Generosity                      2198
Explained by: Perceptions of corruption       2200
Dystopia + residual                           2355
report_year                                      0
Ladder score                                  4232
upperwhisker                                  4232
lowerwhisker                                  4232
Standard error of ladder score                4375
Logged GDP per capita                         4375
Social support                 

## Null values investigation 

Lots of null values here but since lots of rows have the same numbe rof null values it makes me think that its column naming mismatch between the years causing lots of nulls when the concat stuck teh tables together.

So I need to standarse the header namings and see how headers lin eup in each table 


In [90]:
for year, path in years.items():
    df = pd.read_excel(path)
    print(year, list(df.columns))

2026 ['Year', 'Rank', 'Country name', 'Life evaluation (3-year average)', 'Lower whisker', 'Upper whisker', 'Explained by: Log GDP per capita', 'Explained by: Social support', 'Explained by: Healthy life expectancy', 'Explained by: Freedom to make life choices', 'Explained by: Generosity', 'Explained by: Perceptions of corruption', 'Dystopia + residual']
2025 ['Year', 'Rank', 'Country name', 'Life evaluation (3-year average)', 'Lower whisker', 'Upper whisker', 'Explained by: Log GDP per capita', 'Explained by: Social support', 'Explained by: Healthy life expectancy', 'Explained by: Freedom to make life choices', 'Explained by: Generosity', 'Explained by: Perceptions of corruption', 'Dystopia + residual']
2024 ['Country name', 'Ladder score', 'upperwhisker', 'lowerwhisker', 'Explained by: Log GDP per capita', 'Explained by: Social support', 'Explained by: Healthy life expectancy', 'Explained by: Freedom to make life choices', 'Explained by: Generosity', 'Explained by: Perceptions of cor

It's clearly a mess so going to use AI to help me map the right columns together

In [91]:
rename_maps = {
    2026: {
        'Country name': 'country', 'Life evaluation (3-year average)': 'happiness_score',
        'Rank': 'rank', 'Lower whisker': 'whisker_low', 'Upper whisker': 'whisker_high',
        'Explained by: Log GDP per capita': 'gdp_explained',
        'Explained by: Social support': 'social_support_explained',
        'Explained by: Healthy life expectancy': 'life_expectancy_explained',
        'Explained by: Freedom to make life choices': 'freedom_explained',
        'Explained by: Generosity': 'generosity_explained',
        'Explained by: Perceptions of corruption': 'corruption_explained',
        'Dystopia + residual': 'dystopia_residual',
    },
    2025: {  # same shape as 2026
        'Country name': 'country', 'Life evaluation (3-year average)': 'happiness_score',
        'Rank': 'rank', 'Lower whisker': 'whisker_low', 'Upper whisker': 'whisker_high',
        'Explained by: Log GDP per capita': 'gdp_explained',
        'Explained by: Social support': 'social_support_explained',
        'Explained by: Healthy life expectancy': 'life_expectancy_explained',
        'Explained by: Freedom to make life choices': 'freedom_explained',
        'Explained by: Generosity': 'generosity_explained',
        'Explained by: Perceptions of corruption': 'corruption_explained',
        'Dystopia + residual': 'dystopia_residual',
    },
    2024: {
        'Country name': 'country', 'Ladder score': 'happiness_score',
        'upperwhisker': 'whisker_high', 'lowerwhisker': 'whisker_low',
        'Explained by: Log GDP per capita': 'gdp_explained',
        'Explained by: Social support': 'social_support_explained',
        'Explained by: Healthy life expectancy': 'life_expectancy_explained',
        'Explained by: Freedom to make life choices': 'freedom_explained',
        'Explained by: Generosity': 'generosity_explained',
        'Explained by: Perceptions of corruption': 'corruption_explained',
        'Dystopia + residual': 'dystopia_residual',
    },
    2023: {
        'Country name': 'country', 'Ladder score': 'happiness_score',
        'upperwhisker': 'whisker_high', 'lowerwhisker': 'whisker_low',
        'Explained by: Log GDP per capita': 'gdp_explained',
        'Explained by: Social support': 'social_support_explained',
        'Explained by: Healthy life expectancy': 'life_expectancy_explained',
        'Explained by: Freedom to make life choices': 'freedom_explained',
        'Explained by: Generosity': 'generosity_explained',
        'Explained by: Perceptions of corruption': 'corruption_explained',
        'Dystopia + residual': 'dystopia_residual',
    },
    2022: {
        'Country': 'country', 'Happiness score': 'happiness_score',
        'RANK': 'rank', 'Whisker-high': 'whisker_high', 'Whisker-low': 'whisker_low',
        'Explained by: GDP per capita': 'gdp_explained',
        'Explained by: Social support': 'social_support_explained',
        'Explained by: Healthy life expectancy': 'life_expectancy_explained',
        'Explained by: Freedom to make life choices': 'freedom_explained',
        'Explained by: Generosity': 'generosity_explained',
        'Explained by: Perceptions of corruption': 'corruption_explained',
        'Dystopia (1.83) + residual': 'dystopia_residual',
    },
}

standard_cols = ['country', 'happiness_score', 'rank', 'whisker_low', 'whisker_high',
                  'gdp_explained', 'social_support_explained', 'life_expectancy_explained',
                  'freedom_explained', 'generosity_explained', 'corruption_explained',
                  'dystopia_residual', 'report_year']

happiness_frames_v2 = []
for year, path in years.items():
    df = pd.read_excel(path)
    df = df.rename(columns=rename_maps[year])
    df['report_year'] = year
    df = df[[c for c in standard_cols if c in df.columns]]  # keep only standardised columns that exist
    happiness_frames_v2.append(df)

happiness = pd.concat(happiness_frames_v2, ignore_index=True)
happiness.isnull().sum()

country                         0
happiness_score                 1
rank                          280
whisker_low                  2189
whisker_high                 2189
gdp_explained                2198
social_support_explained     2198
life_expectancy_explained    2204
freedom_explained            2201
generosity_explained         2198
corruption_explained         2200
dystopia_residual            2209
report_year                     0
dtype: int64

## Still getting lots of null values???

Also, not sure ho we are getting over 193 null values sicne there are no more countries.... so lets check the values for each 

In [92]:
print(len(happiness))
happiness['report_year'].value_counts()

4512


report_year
2026    2116
2025    1969
2022     147
2024     143
2023     137
Name: count, dtype: int64

Ah, looks like 2025 and 2026 have all the previous years so we dont actually need the other files....

In [93]:
df_2026 = pd.read_excel(years[2026])
df_2026['Year'].unique()

array([2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015,
       2014, 2012, 2011])

## So using AI th Standardise 

In [94]:
happiness = pd.read_excel(years[2026])
happiness = happiness.rename(columns={
    'Country name': 'country',
    'Life evaluation (3-year average)': 'happiness_score',
    'Year': 'report_year',
    'Rank': 'rank',
    'Lower whisker': 'whisker_low',
    'Upper whisker': 'whisker_high',
    'Explained by: Log GDP per capita': 'gdp_explained',
    'Explained by: Social support': 'social_support_explained',
    'Explained by: Healthy life expectancy': 'life_expectancy_explained',
    'Explained by: Freedom to make life choices': 'freedom_explained',
    'Explained by: Generosity': 'generosity_explained',
    'Explained by: Perceptions of corruption': 'corruption_explained',
    'Dystopia + residual': 'dystopia_residual',
})
happiness.isnull().sum()

report_year                     0
rank                            0
country                         0
happiness_score                 0
whisker_low                  1094
whisker_high                 1094
gdp_explained                1097
social_support_explained     1097
life_expectancy_explained    1100
freedom_explained            1099
generosity_explained         1097
corruption_explained         1098
dystopia_residual            1103
dtype: int64

### Still loads of null values but there all simialr numbers still so im assuming somme years just dont have values thats others do

In [95]:
happiness.groupby('report_year')['gdp_explained'].apply(lambda x: x.isnull().sum())

report_year
2011    156
2012    156
2014    158
2015    157
2016    155
2017    156
2018    156
2019      0
2020      0
2021      0
2022      0
2023      3
2024      0
2025      0
Name: gdp_explained, dtype: int64

###  Nice! Now I'm going to get rid of everything pre 2019, we already have 7 years of data 

In [96]:
happiness = happiness[happiness['report_year'] >= 2019].copy()
happiness.isnull().sum()

report_year                  0
rank                         0
country                      0
happiness_score              0
whisker_low                  0
whisker_high                 0
gdp_explained                3
social_support_explained     3
life_expectancy_explained    6
freedom_explained            5
generosity_explained         3
corruption_explained         4
dystopia_residual            9
dtype: int64

### Still some null values so lets have a look at these specifically 

In [97]:
happiness[happiness['gdp_explained'].isnull()]

,report_year,rank,country,happiness_score,whisker_low,whisker_high,gdp_explained,social_support_explained,life_expectancy_explained,freedom_explained,generosity_explained,corruption_explained,dystopia_residual
355,2023,62,Bahrain,5.959,5.766,6.153,NaN,NaN,NaN,NaN,NaN,NaN,NaN
381,2023,88,Tajikistan,5.281,5.201,5.361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
396,2023,103,State of Palestine,4.879,4.753,5.006,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Okay safe to drop them as tehy dont have enough infor to go off


In [98]:
happiness = happiness.dropna(subset=['gdp_explained']).copy()
happiness.isnull().sum()

report_year                  0
rank                         0
country                      0
happiness_score              0
whisker_low                  0
whisker_high                 0
gdp_explained                0
social_support_explained     0
life_expectancy_explained    3
freedom_explained            2
generosity_explained         0
corruption_explained         1
dystopia_residual            6
dtype: int64

In [99]:
happiness[happiness['life_expectancy_explained'].isnull()]

,report_year,rank,country,happiness_score,whisker_low,whisker_high,gdp_explained,social_support_explained,life_expectancy_explained,freedom_explained,generosity_explained,corruption_explained,dystopia_residual
108,2025,109,State of Palestine,4.694,4.588,4.801,1.242,1.335,NaN,0.742,0.073,0.095,NaN
254,2024,108,State of Palestine,4.780,4.675,4.886,1.047,1.456,NaN,0.618,0.055,0.081,NaN
535,2022,99,State of Palestine,4.908,4.727,5.089,1.144,1.309,NaN,0.416,0.065,0.067,NaN


In [100]:
happiness[happiness['dystopia_residual'].isnull()]

,report_year,rank,country,happiness_score,whisker_low,whisker_high,gdp_explained,social_support_explained,life_expectancy_explained,freedom_explained,generosity_explained,corruption_explained,dystopia_residual
87,2025,88,Tajikistan,5.591,5.513,5.669,1.180,1.476,0.783,NaN,0.085,0.336,NaN
108,2025,109,State of Palestine,4.694,4.588,4.801,1.242,1.335,NaN,0.742,0.073,0.095,NaN
198,2024,52,Oman,6.197,5.973,6.420,1.603,1.411,0.500,0.930,0.142,NaN,NaN
236,2024,90,Tajikistan,5.411,5.337,5.485,0.914,1.555,0.658,NaN,0.085,0.342,NaN
254,2024,108,State of Palestine,4.780,4.675,4.886,1.047,1.456,NaN,0.618,0.055,0.081,NaN
535,2022,99,State of Palestine,4.908,4.727,5.089,1.144,1.309,NaN,0.416,0.065,0.067,NaN


### Looking at these data points, and given that we are in a ranked system I think its safe to interpolate these values 

Could do this two ways:

1. interpoalte basd on the countrys ranked above and below
2. interpolate based on that countries data from previous or future years 

I think 2 is better bu for this each country needs at least  two data points to interpolate sufficently so lets check that 

In [101]:
# check if any country had zero non-null 
def has_no_data(series):
    return series.notna().sum() <= 1

cols_to_check = ['life_expectancy_explained', 'freedom_explained', 'corruption_explained', 'dystopia_residual']

for col in cols_to_check:
    empty_countries = happiness.groupby('country')[col].apply(has_no_data).sum()
    print(col, ':', empty_countries, 'countries with less than one data')

life_expectancy_explained : 2 countries with less than one data
freedom_explained : 2 countries with less than one data
corruption_explained : 3 countries with less than one data
dystopia_residual : 3 countries with less than one data


Okay looks like there are a few countires with less than one data point to go off 

lets have a look into one

In [102]:
counts_table = happiness.groupby('country')[cols_to_check].sum()
counts_table

,life_expectancy_explained,freedom_explained,corruption_explained,dystopia_residual
country,,,,
Afghanistan,1.341052,0.000000,0.429226,6.800236
Albania,4.807330,4.557946,0.280361,10.977897
Algeria,4.663419,2.302944,1.083191,11.510812
Argentina,4.369774,4.681840,0.533415,13.326541
Armenia,4.952857,4.377076,1.155618,9.720162
...,...,...,...,...
Venezuela,3.827026,3.273717,0.637625,18.024841
Viet Nam,4.295134,5.779836,0.948848,12.402978
Yemen,2.170000,2.199721,0.602352,6.041473


Looks like some values just have zeros whcih could be skewing the data so lets look into these 

Let check if to see how many values in these columns have 0 as their value 

In [103]:
(happiness[cols_to_check] == 0).sum()

life_expectancy_explained    7
freedom_explained            7
corruption_explained         7
dystopia_residual            0
dtype: int64

Looks like we have a fair few zero values which will be messing up the data 


Actually I've decided that these points are in-siginficnt in the data set and therefore i'm going to leave them

In [104]:
happiness.head()

,report_year,rank,country,happiness_score,whisker_low,whisker_high,gdp_explained,social_support_explained,life_expectancy_explained,freedom_explained,generosity_explained,corruption_explained,dystopia_residual
0,2025,1,Finland,7.764,7.690,7.837,1.915,1.638,0.939,1.105,0.093,0.491,1.582
1,2025,2,Iceland,7.540,7.449,7.630,1.971,1.720,0.996,1.105,0.187,0.187,1.373
2,2025,3,Denmark,7.539,7.446,7.631,1.986,1.633,0.930,1.081,0.125,0.474,1.310
3,2025,4,Costa Rica,7.439,7.356,7.522,1.697,1.483,0.739,1.101,0.059,0.122,2.236
4,2025,5,Sweden,7.255,7.172,7.337,1.950,1.570,1.027,1.070,0.149,0.447,1.041


# Happiness Data is Clean 

Now its time to pull and clean the World banking data (GDP per capita, inflation, unemplyment and Gini index)


